In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

# notebook is in RAG/notebooks → project root is parent
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

PROJECT_ROOT: C:\Users\Vohita\RAG
src exists: True


In [3]:
DATA_PATH = Path.cwd().parent / "data_legal"

In [4]:
import pandas as pd

chunks = pd.read_csv(DATA_PATH / "rag_corpus_chunks.csv")


In [5]:
chunks = chunks.reset_index(drop=True)
chunks["row_id"] = chunks.index

In [6]:
import pickle

with open(DATA_PATH / "indexes" / "bm25_legal.pkl", "rb") as f:
    bm25 = pickle.load(f)


In [9]:
from src.retrieval.tokenization import tokenize

query = "non compete restriction in license agreement"
query_tokens = tokenize(query)

# BM25Index returns list of (index, score)
bm25_results = bm25.search(
    query=query,
    k=5
)

bm25_results


[(8, 8.766641721974768),
 (279, 7.858397601796682),
 (155, 7.823043687014643),
 (11, 7.750779344627694),
 (435, 7.506050471430794)]

In [11]:
for row_id, score in bm25_results:
    print("-" * 80)
    print("row_id:", row_id, "score:", score)
    print(chunks.iloc[row_id]["chunk_text"][:300])

--------------------------------------------------------------------------------
row_id: 8 score: 8.766641721974768
Products. Any failure of Endorser to disclose such conflicting interests, or any breach of this Section, shall be deemed a material breach of the Agreement.', "Endorser's duty not to compete with the business of MusclePharm shall continue for a period of one year following the expiration or terminat
--------------------------------------------------------------------------------
row_id: 279 score: 7.858397601796682
Gulf Oil and Gulf India each agree during the Non-Compete Period not to acquire, directly or indirectly, control of any businesses involved in, or otherwise competing with, the business of the Combined Business from any entity on Schedule 1 hereto.', 'Each Seller agrees that for a period commencing 
--------------------------------------------------------------------------------
row_id: 155 score: 7.823043687014643
Paragraph 1(a) of the Agreement is amended to 

In [13]:
from sentence_transformers import SentenceTransformer

# MUST be the SAME model you used to build embeddings
embedder = SentenceTransformer("all-MiniLM-L12-v2")


In [16]:
import faiss


In [18]:
import faiss
from pathlib import Path

DATA_PATH = Path.cwd().parent / "data_legal"

index = faiss.read_index(
    str(DATA_PATH / "indexes" / "faiss.index")
)

print("FAISS index loaded. Size:", index.ntotal)


FAISS index loaded. Size: 458


In [19]:
# query embedding must be float32 and normalized
query = "non compete restriction in license agreement"

query_embedding = embedder.encode(
    query,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding.reshape(1, -1))

distances, indices = index.search(
    query_embedding.reshape(1, -1),
    k=5
)

list(zip(indices[0], distances[0]))


[(450, 0.45243558),
 (408, 0.44628477),
 (3, 0.4429192),
 (252, 0.4387163),
 (153, 0.42587176)]

In [20]:
# query embedding must be float32 and normalized
query = "non compete restriction in license agreement"

query_embedding = embedder.encode(
    query,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding.reshape(1, -1))

distances, indices = index.search(
    query_embedding.reshape(1, -1),
    k=5
)

list(zip(indices[0], distances[0]))



[(450, 0.45243558),
 (408, 0.44628477),
 (3, 0.4429192),
 (252, 0.4387163),
 (153, 0.42587176)]

In [22]:
import faiss
from pathlib import Path

DATA_PATH = Path.cwd().parent / "data_legal"

index = faiss.read_index(
    str(DATA_PATH / "indexes" / "faiss.index")
)

print("FAISS index loaded. Size:", index.ntotal)



FAISS index loaded. Size: 458


In [23]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(model_name, device="cpu")


In [24]:
import faiss

index = faiss.read_index(str(DATA_PATH / "indexes" / "faiss.index"))


In [26]:
# query embedding must be float32 and normalized
query_embedding = embedder.encode(
    "non compete restriction in license agreement",
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding.reshape(1, -1))

distances, indices = index.search(
    query_embedding.reshape(1, -1),
    k=5
)

list(zip(indices[0], distances[0]))


[(113, 0.6256729),
 (451, 0.62344074),
 (252, 0.61215436),
 (408, 0.5833131),
 (410, 0.58087194)]

In [32]:
def rank_normalize(results):
    """
    results: list of (row_id, score)
    returns: dict {row_id: normalized_score}
    """
    if not results:
        return {}

    scores = [s for _, s in results]
    max_s = max(scores)
    min_s = min(scores)

    if max_s == min_s:
        return {idx: 1.0 for idx, _ in results}

    return {
        idx: (score - min_s) / (max_s - min_s)
        for idx, score in results
    }


def merge_candidates(bm25_norm, faiss_norm, overlap_boost=0.1):
    """
    Merge normalized BM25 + FAISS scores.
    """
    merged = {}

    for idx, score in bm25_norm.items():
        merged[idx] = merged.get(idx, 0.0) + score

    for idx, score in faiss_norm.items():
        merged[idx] = merged.get(idx, 0.0) + score

    # boost overlapping candidates
    overlap = set(bm25_norm) & set(faiss_norm)
    for idx in overlap:
        merged[idx] += overlap_boost

    return merged


def rank_hybrid_candidates(merged_scores, top_k=5):
    """
    Return top-k (row_id, score)
    """
    return sorted(
        merged_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]


In [30]:
query = "non compete restriction in license agreement"

bm25_candidates = bm25.search(
    query=query,
    k=50
)


In [31]:
# embed query
query_embedding = embedder.encode(
    query,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding.reshape(1, -1))

# FAISS search
distances, indices = index.search(
    query_embedding.reshape(1, -1),
    k=50
)

faiss_candidates = list(zip(indices[0], distances[0]))


In [34]:
query = "non compete restriction in license agreement"

# --- BM25 ---
bm25_candidates = bm25.search(
    query=query,
    k=50
)

# --- FAISS ---
query_embedding = embedder.encode(
    query,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding.reshape(1, -1))

faiss_scores, faiss_indices = index.search(
    query_embedding.reshape(1, -1),
    k=50
)

faiss_candidates = list(zip(faiss_indices[0], faiss_scores[0]))


In [35]:
len(bm25_candidates), len(faiss_candidates)

(50, 50)

In [36]:
bm25_norm = rank_normalize(bm25_candidates)
faiss_norm = rank_normalize(faiss_candidates)

In [37]:
list(bm25_norm.items())[:5], list(faiss_norm.items())[:5]

([(8, 1.0),
  (279, 0.80946767467988),
  (155, 0.8020510964241677),
  (11, 0.7868914131552962),
  (435, 0.7355519546911607)],
 [(113, 1.0),
  (451, 0.9840713),
  (252, 0.90353096),
  (408, 0.69771767),
  (410, 0.6802974)])

In [38]:
merged_scores = merge_candidates(
    bm25_norm,
    faiss_norm,
    overlap_boost=0.1
)

In [39]:
len(merged_scores)

83

In [40]:
hybrid_results = rank_hybrid_candidates(
    merged_scores,
    top_k=5
)

hybrid_results

[(408, 1.3990559768429391),
 (3, 1.3358491411415103),
 (450, 1.2601770927647455),
 (126, 1.2496163054212417),
 (421, 1.2297810738173185)]

In [41]:
for row_id, score in hybrid_results:
    print("-" * 80)
    print("row_id:", row_id, "score:", score)
    print(chunks.iloc[row_id]["chunk_text"][:300])

--------------------------------------------------------------------------------
row_id: 408 score: 1.3990559768429391
"Subject to the terms of this Agreement, subject to Manufacturer meeting EMV's requirements for quality, price and lead- time, EMV hereby grants Manufacturer an exclusive, non-transferable, license (without the right to sublicense) under EMV's Proprietary Rights in the Territory, during the term of 
--------------------------------------------------------------------------------
row_id: 3 score: 1.3358491411415103
"Subject to Licensee's on\xadgoing compliance with Section 3.2 and all other terms and conditions of this Agreement, Licensor grants to Licensee an exclusive (save for rights reserved to Licensor hereunder), non-transferable (except as provided in Section 11.7) and non- sublicensable license, during
--------------------------------------------------------------------------------
row_id: 450 score: 1.2601770927647455
"Licensor and Licensee do hereby agree that

In [45]:
# ================================
# 1. Imports
# ================================
import faiss
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ================================
# 2. Paths
# ================================
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data_legal"

# ================================
# 3. Load data
# ================================
chunks = pd.read_csv(DATA_PATH / "rag_corpus_chunks.csv")

# ================================
# 4. Load BM25
# ================================
with open(DATA_PATH / "indexes" / "bm25_legal.pkl", "rb") as f:
    bm25 = pickle.load(f)

# ================================
# 5. Load FAISS
# ================================
index = faiss.read_index(
    str(DATA_PATH / "indexes" / "faiss.index")
)

# ================================
# 6. Load Embedder
# ================================
embedder = SentenceTransformer("all-MiniLM-L12-v2")

# ================================
# 7. Query
# ================================
query = "non compete restriction in license agreement"

# ================================
# 8. BM25 Retrieval
# ================================
bm25_results = bm25.search(
    query=query,
    k=50
)
# bm25_results = [(row_id, score), ...]

# ================================
# 9. FAISS Retrieval
# ================================
query_embedding = embedder.encode(
    query,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding.reshape(1, -1))

faiss_scores, faiss_indices = index.search(
    query_embedding.reshape(1, -1),
    k=50
)

faiss_results = list(zip(faiss_indices[0], faiss_scores[0]))

# ================================
# 10. Simple Hybrid Merge
# ================================
hybrid_scores = {}

for row_id, score in bm25_results:
    hybrid_scores[row_id] = hybrid_scores.get(row_id, 0) + score

for row_id, score in faiss_results:
    hybrid_scores[row_id] = hybrid_scores.get(row_id, 0) + score

hybrid_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

# ================================
# 11. Print Results + Metadata
# ================================
for row_id, score in hybrid_results:
    print("-" * 80)
    print("row_id:", row_id, "| score:", score)
    print("chunk_id:", chunks.iloc[row_id]["chunk_id"])
    print("doc_id:", chunks.iloc[row_id]["doc_id"])
    print("domain:", chunks.iloc[row_id]["domain"])
    print()
    print(chunks.iloc[row_id]["chunk_text"][:400])


--------------------------------------------------------------------------------
row_id: 8 | score: 8.766641721974768
chunk_id: LC00009
doc_id: LEGAL_DOC_001
domain: legal

Products. Any failure of Endorser to disclose such conflicting interests, or any breach of this Section, shall be deemed a material breach of the Agreement.', "Endorser's duty not to compete with the business of MusclePharm shall continue for a period of one year following the expiration or termination of this Agreement.", "Endorser's duty not to compete with the business of MusclePharm shall cont
--------------------------------------------------------------------------------
row_id: 155 | score: 8.178328765301767
chunk_id: LC00156
doc_id: LEGAL_DOC_016
domain: legal

Paragraph 1(a) of the Agreement is amended to provide that Fox grants Licensee a worldwide, exclusive (except as otherwise may be provided in the Agreement), non-transferable right and license to distribute video clips for the property "KINGDOM OF HEA

In [27]:
# # Hybrid Retrieval + ABAC
# # metadata dataframe already loaded earlier
# metadata_store = MetadataStore(metadata)

# # single lookup
# metadata_store.get(42)

# # batch lookup (e.g. hybrid results)
# row_ids = [idx for idx, _ in hybrid_results]
# metadata_store.batch_get(row_ids)


NameError: name 'MetadataStore' is not defined

In [28]:
# class MetadataStore:
#     def __init__(self, metadata_df, key="row_id"):
#         self.key = key
#         self._store = (
#             metadata_df
#             .set_index(key)
#             .to_dict(orient="index")
#         )

#     def get(self, row_id):
#         return self._store.get(row_id, None)

#     def batch_get(self, row_ids):
#         return [self._store.get(rid, None) for rid in row_ids]


In [29]:
# metadata_store = MetadataStore(metadata)

# metadata_store.get(42)


{'chunk_id': 'C00043',
 'security_tier_level': 0,
 'owner_team': 'legal',
 'domain': 'support_faq'}

In [31]:
# metadata.head()


,row_id,chunk_id,security_tier_level,owner_team,domain
0,0,C00001,1,legal,support_faq
1,1,C00002,1,legal,support_faq
2,2,C00003,1,legal,support_faq
3,3,C00004,1,legal,support_faq
4,4,C00005,1,legal,support_faq


In [32]:
# # User context (same intent)
# class UserContext:
#     def __init__(self, user_id, department, clearance, projects=None):
#         self.user_id = user_id
#         self.department = department
#         self.clearance = clearance
#         self.projects = projects or []


# # ABAC check (same intent as before)
# def abac_allows(row, user):
#     """
#     ABAC decision based on chunk metadata row and user context.
#     """
#     return (
#         row["security_tier_level"] <= user.clearance
#         and row["owner_team"] == user.department
#     )


In [34]:
# user = UserContext(
#     user_id="u123",
#     department="legal",
#     clearance=3
# )

# # get a metadata row first
# metadata_row = metadata_store.get(0)   # any valid row_id

# abac_allows(metadata_row, user)


True

In [35]:
# class HybridRetriever:
#     def __init__(
#         self,
#         bm25,
#         faiss_index,
#         embedder,
#         metadata_store,
#         overlap_boost=0.1
#     ):
#         self.bm25 = bm25
#         self.faiss_index = faiss_index
#         self.embedder = embedder
#         self.metadata_store = metadata_store
#         self.overlap_boost = overlap_boost

#     def retrieve(self, query, top_k=10):
#         # --- BM25 ---
#         bm25_idx, bm25_scores = self.bm25.search(query=query, k=50)
#         bm25_candidates = list(zip(bm25_idx, bm25_scores))
#         bm25_norm = rank_normalize(bm25_candidates)

#         # --- FAISS ---
#         query_embedding = self.embedder.encode(
#             query,
#             convert_to_numpy=True
#         ).astype("float32")

#         faiss.normalize_L2(query_embedding.reshape(1, -1))

#         faiss_scores, faiss_idx = self.faiss_index.search(
#             query_embedding.reshape(1, -1),
#             k=50
#         )
#         faiss_candidates = list(zip(faiss_idx[0], faiss_scores[0]))
#         faiss_norm = rank_normalize(faiss_candidates)

#         # --- Hybrid merge ---
#         merged = merge_candidates(
#             bm25_norm,
#             faiss_norm,
#             overlap_boost=self.overlap_boost
#         )

#         return rank_hybrid_candidates(merged, top_k=top_k)


In [36]:
# hybrid_retriever = HybridRetriever(
#     bm25=bm25,
#     faiss_index=index,
#     embedder=embedder,
#     metadata_store=metadata_store
# )

# hybrid_results = hybrid_retriever.retrieve(
#     "authentication timeout",
#     top_k=5
# )

# hybrid_results


[(1242, 1.6413162275039967),
 (1237, 1.6175782370248024),
 (2758, 1.5232522773423378),
 (4203, 1.5070183999496558),
 (1305, 1.4765056296074208)]

In [38]:
# for row_id, score in low_results:
#     meta = metadata_store.get(row_id)
#     assert meta["security_tier_level"] <= low_user.clearance
#     assert meta["owner_team"] == low_user.department


NameError: name 'low_results' is not defined

In [39]:
# low_user = UserContext(
#     user_id="u_low",
#     department="legal",
#     clearance=1
# )

# low_results = hybrid_retriever.retrieve(
#     query="authentication timeout",
#     top_k=20
# )

In [40]:
# metadata_store.get(row_id)


{'chunk_id': 'C01306',
 'security_tier_level': 0,
 'owner_team': 'product',
 'domain': 'developer_docs'}

In [42]:
# low_results_secure = [
#     (row_id, score)
#     for row_id, score in low_results
#     if abac_allows(metadata_store.get(row_id), low_user)
# ]


In [43]:
# for row_id, score in low_results_secure:
#     meta = metadata_store.get(row_id)
#     assert meta["security_tier_level"] <= low_user.clearance
#     assert meta["owner_team"] == low_user.department


In [45]:
# high_results = hybrid_retriever.retrieve(
#     query=query,
#     top_k=5
# )

# high_results


[(1242, 1.6413162275039967),
 (1237, 1.6175782370248024),
 (2758, 1.5232522773423378),
 (4203, 1.5070183999496558),
 (1305, 1.4765056296074208)]

In [46]:
# print("Low clearance results:", len(low_results))
# print("High clearance results:", len(high_results))

Low clearance results: 20
High clearance results: 5


In [47]:
# assert len(high_results) >= len(low_results)

AssertionError: 

In [48]:
# CANDIDATE_K = 100  # larger than final K

# raw_results = hybrid_retriever.retrieve(
#     query=query,
#     top_k=CANDIDATE_K
# )


In [49]:
# secure_candidates = [
#     (rid, score)
#     for rid, score in raw_results
#     if abac_allows(metadata_store.get(rid), user)
# ]


In [50]:
# final_results = secure_candidates[:5]


In [51]:
# assert len(high_results_secure) >= len(low_results_secure)
# # 

NameError: name 'high_results_secure' is not defined

In [53]:
# high_user = UserContext(
#     user_id="u_high",
#     department="legal",
#     clearance=4   # higher than low_user
# )


In [54]:
# # get raw results first
# high_results = hybrid_retriever.retrieve(
#     query=query,
#     top_k=100   # larger pool before ABAC
# )

# # apply ABAC
# high_results_secure = [
#     (rid, score)
#     for rid, score in high_results
#     if abac_allows(metadata_store.get(rid), high_user)
# ]


In [55]:
# low_user = UserContext(
#     user_id="u_low",
#     department="legal",
#     clearance=1
# )


In [56]:
# len(high_results_secure)


14

In [58]:
# for row_id, score in high_results:
#     print("-" * 80)
#     print("row_id:", row_id, "score:", score)
#     print(chunks.iloc[row_id]["chunk_text"][:300])

--------------------------------------------------------------------------------
row_id: 1242 score: 1.6413162275039967
This portion is taken from developer documentation used by engineering teams. In this section of 'Service-to-service auth and authentication — engineering guide 2024' focuses on how engineering teams use the API in day‑to‑day work. It covers authentication patterns, request and response shapes, and 
--------------------------------------------------------------------------------
row_id: 1237 score: 1.6175782370248024
Here is a technical section designed for engineers integrating with a system. This opening section of 'Service-to-service auth and authentication — engineering guide 2024' focuses on how engineering teams use the API in day‑to‑day work. It covers authentication patterns, request and response shapes,
--------------------------------------------------------------------------------
row_id: 2758 score: 1.5232522773423378
This chunk contains part of a develope

In [60]:
query = "CEO personal phone number"

results = hybrid_retriever.retrieve(
    query=query,
    top_k=50   # get a larger pool first
)


In [61]:
query = "CEO personal phone number"

results = hybrid_retriever.retrieve(
    query=query,
    top_k=50   # get a larger pool first
)


In [63]:
query = "CEO personal phone number"

# retrieve a larger candidate pool
results = hybrid_retriever.retrieve(
    query=query,
    top_k=50
)

# apply ABAC for low-clearance user
secure_results = [
    (rid, score)
    for rid, score in results
    if abac_allows(metadata_store.get(rid), low_user)
][:5]


In [64]:
secure_results


[(1342, 0.2971663773059845),
 (1339, 0.1857348531484604),
 (1336, 0.04897843673825264)]